In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Fixed channel-margin continuation: three new updates

Run all starts from the actual AFTER/terminal.pt saved in run20261001T133512881485Z. POINT1 replays that existing step1; POINT2–POINT4 apply exactly three additional fresh-gradient updates. The primary endpoint is fixed POINT4. All intermediate successes, regressions, zero-update outcomes and failures remain in the result. No best-iterate selection or stopping when rank1 is first reached.

Unchanged: original code/basis/write support, receiver g0/R44/all174valid candidates, public template-derived margin loss, eta174 and per-step L2cap1. Each update uses its own current terminal→native VAE decode→RGB8→actual materialized YUV444→RGB24→native VAE encode. Encoder and decoder VJPs are recomputed every time. The identity STE supplies a direction only; actual roundtrip observations determine Delta/rank. There is no new total-displacement clipping: new-step L2 sums are bounded by3 in ideal arithmetic; cumulative actual change from the original marked terminal is reported separately.

Four new full observations, eight path/payload reads, sixteen message comparisons and1392valid costs. Historical OFF/PAYLOAD_MULTI/OVERLAP_MULTI controls and the prior step1 are preserved as references, not independent new negatives. Quality is reported against the original marked terminal's roundtrip, the saved step1 and the preceding point; none is an unmarked-source quality certificate.

The previous real run used about13.3minutes for one gradient pair,19.35GiB peak allocated CUDA memory and64.53GiB temporary disk. Three pairs suggest roughly40minutes plus media/model-loading/persistence overhead, not a runtime guarantee. Both gradient phases run separately and clean their temporary storage. No GPU-model gate; use the successful current-Python setup and a fresh child. No Transformer or generation replay, no H264/MP4, no unknown-phase claim.

This fixed same-source development diagnostic has local CPU/static validation. Real pretrained VAE/FFmpeg execution is for the user. Source is immutable when SOURCE_SHA is bound; an unbound draft is not a released notebook.


In [ ]:
SOURCE_SHA = '14589a13472f62ad83165d3f1e23efdd15e75d61'
from pathlib import Path
import datetime,json,os,subprocess,sys,traceback
STAMP=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=Path('/content/drive/MyDrive/Video-WM/Zero-Mean-Channel-Margin-Loop-V1')/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
REPO=Path('/content/SC-SSTW-CHANNEL-MARGIN-LOOP-'+STAMP)
RUN_OUTPUT=OUTPUT/'fixed_reference'
PYTHON=sys.executable
INPUT_ROOT=Path('/content/drive/MyDrive/Video-WM/Zero-Mean-Channel-Margin-V1/20261001T133512881485Z/fixed_reference')
REFERENCE_ROOT=Path('/content/drive/MyDrive/Video-WM/Zero-Mean-C1-YUV444-No-H264-V1/20261001T110124499275Z/fixed_reference')
setup=dict(status='SETUP_STARTED',source_sha=SOURCE_SHA, fixed_denominator={'new_updates': 3, 'observations': 4, 'path_reads': 8, 'payload_reads': 8, 'message_evaluations': 16, 'valid_costs': 1392, 'catalog_slots': 31320, 'structural_exclusions': 29928, 'reference_normalized': 3, 'reference_raw': 6, 'parent_normalized': 1, 'parent_raw': 2}, observations={p:{'status':'NOT_RUN_SETUP'} for p in ('POINT1','POINT2','POINT3','POINT4')})
def write_json(path,value):
    temp=path.with_suffix(path.suffix+'.tmp');temp.write_text(json.dumps(value,indent=2)+'\n');os.replace(temp,path)
def failed(stage,exc):
    setup.update(status='SETUP_FAILED',stage=stage,error=f'{type(exc).__name__}: {exc}')
    write_json(OUTPUT/'setup_receipt.json',setup)
    write_json(OUTPUT/'setup_failure.json',dict(stage=stage,error=str(exc),traceback=traceback.format_exc()))
def logged(command,stage,cwd=None,check=True):
    try:
        with (OUTPUT/'execution.log').open('a') as log:
            log.write('COMMAND '+repr(command)+'\n');log.flush()
            child=subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            for line in child.stdout:print(line,end='',flush=True);log.write(line);log.flush()
            code=child.wait();log.write('EXIT '+str(code)+'\n')
        if check and code:raise subprocess.CalledProcessError(code,command)
        return code
    except Exception as exc:failed(stage,exc);raise
write_json(OUTPUT/'setup_receipt.json',setup)
print('Full writer terminal:',INPUT_ROOT,'Saved references:',REFERENCE_ROOT,'Output:',OUTPUT)
if SOURCE_SHA is None:
    error=RuntimeError('Unpublished draft: bind verified published source SHA before Run all')
    failed('SOURCE_BINDING',error);raise error


In [ ]:
try:
    logged(['git','clone','--filter=blob:none','https://github.com/RICHAAARC/SC-SSTW.git',str(REPO)],'SOURCE_CLONE')
    logged(['git','-C',str(REPO),'fetch','origin',SOURCE_SHA],'SOURCE_FETCH')
    logged(['git','-C',str(REPO),'checkout','--detach',SOURCE_SHA],'SOURCE_CHECKOUT')
    actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True)
    if actual!=SOURCE_SHA or dirty:raise RuntimeError('source SHA/clean checkout mismatch')
    write_json(OUTPUT/'source_receipt.json',dict(source_sha=actual,clean=True))
    torch_ok=logged([PYTHON,'-c','import torch,torchvision; print(torch.__version__,torchvision.__version__)'],'TORCH_PROBE',check=False)==0
    if not torch_ok:
        logged([PYTHON,'-m','pip','install','--upgrade','torch','torchvision'],'TORCH_REPAIR')
    logged([PYTHON,'-m','pip','install','-r',str(REPO/'experiments/wan_state_clock/requirements-grow-video-reference.txt')],'DEPENDENCIES')
    dependency_code=logged([PYTHON,'-m','pip','check'],'DEPENDENCY_REPORT',check=False)
    setup['dependency_check_returncode']=dependency_code
    if dependency_code:print('Metadata conflicts retained; actual imports determine availability.')
    (OUTPUT/'environment_freeze.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
    import_code='import json,torch; from diffusers import AutoencoderKLWan; from experiments.wan_state_clock.zero_mean_c1_yuv444_no_h264_run import environment; r=environment(); r.update(cuda_available=torch.cuda.is_available()); print(json.dumps(r))'
    env=json.loads(subprocess.check_output([PYTHON,'-c',import_code],cwd=REPO,text=True))
    write_json(OUTPUT/'environment_receipt.json',env)
    if not env['cuda_available']:print('CUDA unavailable: CPU path selected; much slower.')
    logged(['ffmpeg','-version'],'FFMPEG_PROBE');logged(['ffprobe','-version'],'FFPROBE_PROBE')
    setup.update(status='SETUP_COMPLETE',python_executable=PYTHON,environment_strategy='current_python_fresh_children')
    write_json(OUTPUT/'setup_receipt.json',setup)
except Exception as exc:failed('SOURCE_OR_ENVIRONMENT',exc);raise


In [ ]:
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN')
    if token:os.environ['HF_TOKEN']=token
except Exception:
    pass
finally:
    token=None
command=[PYTHON,'-u','-m','experiments.wan_state_clock.zero_mean_channel_margin_loop_run','--output',str(RUN_OUTPUT),'--input-root',str(INPUT_ROOT),'--reference-root',str(REFERENCE_ROOT)]
try:
    returncode=logged(command,'FIXED_RUN',cwd=REPO,check=False)
    RESULT_PATH=RUN_OUTPUT/'result.json'
    write_json(OUTPUT/'execution_receipt.json',dict(command=command,returncode=returncode,result_path=str(RESULT_PATH),result_exists=RESULT_PATH.is_file()))
    if not RESULT_PATH.is_file():raise RuntimeError('No runner result; full setup denominator is retained')
    setup.update(status='SUPERSEDED_BY_RESULT',result_path=str(RESULT_PATH));write_json(OUTPUT/'setup_receipt.json',setup)
except Exception as exc:failed('RUNNER',exc);raise


In [ ]:
result=json.loads(RESULT_PATH.read_text())
print('Status:',result['status'],'Counts:',result.get('counts'),'Calls:',result['calls'])
print('Baseline replay:',result.get('baseline_replay'))
for point,row in result['comparisons'].items():print(point,row)
print('Fixed endpoint:',result.get('fixed_endpoint'))
for name,row in result['updates'].items():
    print(name,row['status'],'update=',row.get('receipt'),'sum_new_step_l2=',row.get('sum_new_step_l2'),
          'original_terminal_displacement_l2=',row.get('original_terminal_displacement_l2'))
for name,row in result['message_evaluations'].items():print(name,row)
for point,row in result['observations'].items():
    print(point,'quality_vs_original_marked=',row.get('quality_vs_original_marked'),
          'quality_vs_step1=',row.get('quality_vs_step1'),'quality_vs_previous=',row.get('quality_vs_previous_point'))
print('Resources:',result['resources'])
print('Failures:',result['failures'])
print(result['evidence_ceiling'])
print('Retained result:',RESULT_PATH)
